In [ ]:
import numpy as np
# Lập hệ m x m phương trình
def  lap_he (m):
    
    n = m * m
    A = np.zeros((n, n))
    b = np.zeros(n)

    for i in range(m):              # i = 0 là hàng sát cạnh trên
        for j in range(m):
            k = i * m + j           
            A[k, k] = 4
            if i > 0:     A[k, k - m] = -1   # lân cận phía trên
            else:         b[k] += 100        
            if i < m - 1: A[k, k + m] = -1   # lân cận phía dưới
            if j > 0:     A[k, k - 1] = -1   # lân cận bên trái
            if j < m - 1: A[k, k + 1] = -1   # lân cận bên phải

    print(f"Hệ {n} phương trình, {n} ẩn (m = {m})")
    print(f"\nMa trận A ({n} x {n}):")
    for hang in A:
        print(" ".join(f"{int(v):2d}" for v in hang))
    print(f"\nVế phải b ({n} phần tử):")
    print(" ".join(f"{int(v):3d}" for v in b))

    return A, b

Hệ 9 phương trình, 9 ẩn (m = 3)

Ma trận A (9 x 9):
 4 -1  0 -1  0  0  0  0  0
-1  4 -1  0 -1  0  0  0  0
 0 -1  4  0  0 -1  0  0  0
-1  0  0  4 -1  0 -1  0  0
 0 -1  0 -1  4 -1  0 -1  0
 0  0 -1  0 -1  4  0  0 -1
 0  0  0 -1  0  0  4 -1  0
 0  0  0  0 -1  0 -1  4 -1
 0  0  0  0  0 -1  0 -1  4

Vế phải b (9 phần tử):
100 100 100   0   0   0   0   0   0


In [11]:
import numpy as np

def dinh_thuc(M, tol=1e-12): 
   
    M = np.array(M, dtype=float)  # tạo bản sao để không làm thay đổi ma trận gốc

    if M.ndim != 2 or M.shape[0] != M.shape[1]: # kiểm tra: ma trận là ma trận vuông nếu M là mảng 2 chiều và M có số hàng bằng số cột 
        raise ValueError("Định thức chỉ tính cho ma trận vuông.") 

    # khởi tạo
    n = M.shape[0]
    det = 1.0
    
    for i in range(n):
        # Chọn phần tử chính (phần tử có giá trị max) theo cột
        p = i + np.argmax(np.abs(M[i:, i]))
        
        # nếu phần tử trụ quá nhỏ -> trụ xấp xỉ 0 -> suy biến
        if abs(M[p, i]) < tol:
            return 0.0

        # Đổi dòng thì định thức đổi dấu
        if p != i:
            M[[i, p]] = M[[p, i]]
            det = -det

        # tích các phần tử trụ (phần tử thuộc đường chéo chính)
        det *= M[i, i]

        # Khử các phần tử phía dưới phần tử chính (khử gauss)
        for j in range(i + 1, n):
            he_so = M[j, i] / M[i, i]
            M[j, i:] -= he_so * M[i, i:]

    return det


def conjugate_gradient(A, b, x0, eps) -> tuple[np.ndarray, int]:
    """
    A: Ma trận vuông, đối xứng xác định dương.
    b: Vectơ vế phải.
    x0: Vectơ nghiệm ban đầu.
    eps: Sai số cho phép, eps > 0.
    """

    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float)
    x = np.asarray(x0, dtype=float).copy()

    # Kiểm tra ma trận vuông
    if A.ndim != 2 or A.shape[0] != A.shape[1] or A.shape[0] == 0:
        raise ValueError("A phải là ma trận vuông không rỗng.")

    # lưu cấp của A vào n
    n = A.shape[0]

    # Kiểm tra ma trận đối xứng
    if not np.allclose(A, A.T): # so sánh bằng giữ A vs A^T
        raise ValueError("A phải là ma trận đối xứng.")

    # Kiểm tra xác định dương 
    for k in range(1, n + 1):
        det_k = dinh_thuc(A[:k, :k]) # tính định thức con lấy ra từ k hàng và cột của A
        if det_k <= 0:
            raise ValueError("A không phải ma trận xác định dương ")

    # Khởi tạo phần dư (r) và hướng tìm kiếm (s)
    r = b - np.dot(A, x)
    s = r.copy()

    print("Nghiệm ban đầu x0:", x0)
    print("Phần dư ban đầu r0:", r)

    # kiểm tra nghiệm ban đầu 
    if np.sqrt(np.dot(r, r)) < eps:
        print("Nghiệm ban đầu đã thỏa mãn điều kiện dừng.")
        return x, 0

    # vòng lặp
    for k in range(1, n + 1):

        # Tính alpha = s.r/s.As
        As = np.dot(A, s)
        mau = np.dot(s, As)
        if mau <= 0: # vì ma trận xác định dương nên mẫu phải dương
            raise ValueError("Mẫu số không hợp lệ.")

        alpha = np.dot(s, r) / mau

        # Cập nhật nghiệm, phần dư và chuẩn phần dư để xét điều kiện dừng
        x = x + alpha * s
        r_moi = b - np.dot(A, x)
        chuan_r = np.sqrt(np.sum(r_moi**2))

        # Hiển thị kết quả từng lần lặp
        print(f"\nLần lặp thứ {k}:")
        print("Nghiệm x =", [round(float(i), 6) for i in x])
        print("Phần dư r =", [round(float(i), 6) for i in r_moi])
        print("Chuẩn phần dư ||r|| =", round(float(chuan_r), 6))

        # Kiểm tra điều kiện dừng
        if chuan_r < eps:
            print(f"Thỏa mãn sau {k} lần lặp.")
            return x, k
            
        # Tính beta
        beta = -np.dot(r_moi, As) / mau

        # Cập nhật hướng tìm kiếm
        s = r_moi + beta * s
        r = r_moi.copy()
        
    return x, n

In [12]:
def giai_bai_toan_nhiet(m, eps=1e-6):
    A, b = lap_he(m)

    # Nghiệm ban đầu
    x0 = np.zeros(m * m)

    # Giải bằng Conjugate Gradient
    x, so_lan_lap = conjugate_gradient(A, b, x0, eps)

    # Chuyển nghiệm thành ma trận nhiệt độ
    T = x.reshape(m, m)
    print(f"\nKẾT QUẢ VỚI m = {m}")
    print("Ma trận nhiệt độ:")
    print(np.round(T, 4))
    print("Số lần lặp:", so_lan_lap)
    return T, so_lan_lap

In [14]:
A, b = lap_he(3)
giai_bai_toan_nhiet(3)


Hệ 9 phương trình, 9 ẩn (m = 3)

Ma trận A (9 x 9):
 4 -1  0 -1  0  0  0  0  0
-1  4 -1  0 -1  0  0  0  0
 0 -1  4  0  0 -1  0  0  0
-1  0  0  4 -1  0 -1  0  0
 0 -1  0 -1  4 -1  0 -1  0
 0  0 -1  0 -1  4  0  0 -1
 0  0  0 -1  0  0  4 -1  0
 0  0  0  0 -1  0 -1  4 -1
 0  0  0  0  0 -1  0 -1  4

Vế phải b (9 phần tử):
100 100 100   0   0   0   0   0   0
Nghiệm ban đầu x0: [0. 0. 0. 0. 0. 0. 0. 0. 0.]
Phần dư ban đầu r0: [100. 100. 100.   0.   0.   0.   0.   0.   0.]

Lần lặp thứ 1:
Nghiệm x = [37.5, 37.5, 37.5, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Phần dư r = [-12.5, 25.0, -12.5, 37.5, 37.5, 37.5, 0.0, 0.0, 0.0]
Chuẩn phần dư ||r|| = 71.807033

Lần lặp thứ 2:
Nghiệm x = [39.240506, 53.164557, 39.240506, 13.924051, 13.924051, 13.924051, 0.0, 0.0, 0.0]
Phần dư r = [10.126582, -20.253165, 10.126582, -2.531646, 25.316456, -2.531646, 13.924051, 13.924051, 13.924051]
Chuẩn phần dư ||r|| = 43.019356

Lần lặp thứ 3:
Nghiệm x = [42.613636, 51.704545, 42.613636, 17.045455, 25.0, 17.045455, 3.977273, 3.9

(array([[42.85714286, 52.67857143, 42.85714286],
        [18.75      , 25.        , 18.75      ],
        [ 7.14285714,  9.82142857,  7.14285714]]),
 5)

In [ ]:
A, b = lap_he(5)
giai_bai_toan_nhiet(5)

Hệ 25 phương trình, 25 ẩn (m = 5)

Ma trận A (25 x 25):
 4 -1  0  0  0 -1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
-1  4 -1  0  0  0 -1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
 0 -1  4 -1  0  0  0 -1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
 0  0 -1  4 -1  0  0  0 -1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0 -1  4  0  0  0  0 -1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
-1  0  0  0  0  4 -1  0  0  0 -1  0  0  0  0  0  0  0  0  0  0  0  0  0  0
 0 -1  0  0  0 -1  4 -1  0  0  0 -1  0  0  0  0  0  0  0  0  0  0  0  0  0
 0  0 -1  0  0  0 -1  4 -1  0  0  0 -1  0  0  0  0  0  0  0  0  0  0  0  0
 0  0  0 -1  0  0  0 -1  4 -1  0  0  0 -1  0  0  0  0  0  0  0  0  0  0  0
 0  0  0  0 -1  0  0  0 -1  4  0  0  0  0 -1  0  0  0  0  0  0  0  0  0  0
 0  0  0  0  0 -1  0  0  0  0  4 -1  0  0  0 -1  0  0  0  0  0  0  0  0  0
 0  0  0  0  0  0 -1  0  0  0 -1  4 -1  0  0  0 -1  0  0  0  0  0  0  0  0
 0  0  0  0  0  0  0 -1  0  0  0 -1  4 -1  0

(array([[46.86868687, 62.92249417, 66.94250194, 62.92249417, 46.86868687],
        [24.5522533 , 37.87878788, 41.92501943, 37.87878788, 24.5522533 ],
        [13.46153846, 22.11538462, 25.        , 22.11538462, 13.46153846],
        [ 7.17851593, 12.12121212, 13.84421134, 12.12121212,  7.17851593],
        [ 3.13131313,  5.3467366 ,  6.13442113,  5.3467366 ,  3.13131313]]),
 11)

In [16]:
A, b = lap_he(10)
giai_bai_toan_nhiet(10)

Hệ 100 phương trình, 100 ẩn (m = 10)

Ma trận A (100 x 100):
 4 -1  0  0  0  0  0  0  0  0 -1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
-1  4 -1  0  0  0  0  0  0  0  0 -1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
 0 -1  4 -1  0  0  0  0  0  0  0  0 -1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
 0  0 -1  4 -1  0  0  0  0  0  0  0  0 

(array([[49.08655503, 67.89012925, 76.11798622, 79.95189886, 81.5178176 ,
         81.5178176 , 79.95189886, 76.11798622, 67.89012925, 49.08655503],
        [28.45609096, 46.35597579, 56.6299168 , 62.17179159, 64.60155382,
         64.60155382, 62.17179159, 56.6299168 , 46.35597579, 28.45609096],
        [18.38183306, 32.44776613, 41.87391356, 47.50379686, 50.11505238,
         50.11505238, 47.50379686, 41.87391356, 32.44776613, 18.38183306],
        [12.62347513, 23.17934204, 30.91417444, 35.85442994, 38.23980649,
         38.23980649, 35.85442994, 30.91417444, 23.17934204, 12.62347513],
        [ 8.93272547, 16.7319525 , 22.74901215, 26.75994197, 28.74993713,
         28.74993713, 26.75994197, 22.74901215, 16.7319525 ,  8.93272547],
        [ 6.37547428, 12.06673037, 16.58997975, 19.68638867, 21.25006288,
         21.25006288, 19.68638867, 16.58997975, 12.06673037,  6.37547428],
        [ 4.50244124,  8.56951494, 11.85778778, 14.14557005, 15.31386289,
         15.31386289, 14.1455700

In [17]:
A, b = lap_he(20)
giai_bai_toan_nhiet(20)

Hệ 400 phương trình, 400 ẩn (m = 20)

Ma trận A (400 x 400):
 4 -1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0 -1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0 

(array([[49.75117656, 69.26410205, 78.3003388 , 83.11880278, 85.96875452,
         87.76962425, 88.94371066, 89.7035557 , 90.16202691, 90.37838652,
         90.37838652, 90.16202691, 89.7035557 , 88.94371066, 87.76962425,
         85.96875452, 83.11880278, 78.3003388 , 69.26410205, 49.75117656],
        [29.74060417, 49.00489283, 60.81845035, 68.20611783, 72.98659104,
         76.16603187, 78.30166268, 79.70848525, 80.56616542, 80.97313265,
         80.97313265, 80.56616542, 79.70848525, 78.30166268, 76.16603187,
         72.98659104, 68.20611783, 60.81845035, 49.00489283, 29.74060417],
        [20.20634729, 36.19641472, 47.76245195, 55.90062716, 61.60545997,
         65.60624949, 68.38842295, 70.26255719, 71.42101686, 71.97484598,
         71.97484598, 71.42101686, 70.26255719, 68.38842295, 65.60624949,
         61.60545997, 55.90062716, 47.76245195, 36.19641472, 20.20634729],
        [14.88837025, 27.81196678, 38.13431559, 46.02847889, 51.92837224,
         56.2650832 , 59.38322246, 

In [26]:
ket_qua = [(3, 5), (5, 11), (10, 29), (20, 60)]

print("BẢNG SỐ BƯỚC LẶP")
print("m\tSố bước lặp")
for m, so_lan_lap in ket_qua:
    print(f"{m}\t{so_lan_lap}")

BẢNG SỐ BƯỚC LẶP
m	Số bước lặp
3	5
5	11
10	29
20	60
